## Search Engine with Tools and Agents

In [1]:
## Arxiv - Research
## Tools creaation
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
import os

c:\Users\matcg\OneDrive\Documents\GitHub\LangChain Projects\4-Search Engine\venv_se\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


### Built-in Tools

In [12]:
# Use the in built tool of Wikipedia
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [ ]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arvix = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arvix.name

'arxiv'

In [14]:
tools = [wiki, arvix] # combining the tools.

### Custom Tools [RAG Tool]

In [2]:
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\matcg\OneDrive\Documents\GitHub\LangChain Projects\4-Search Engine\venv_se\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents, OpenAIEmbeddings())
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024BF51D23C0>, search_kwargs={})

In [7]:
# Now i nedd to convert the retriever into a tool.
from langchain_classic.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever=retriever, 
    name="LangSmithDocs",
    description="useful for when you need to answer questions about LangSmithDocs"
)

retriever_tool.name

'LangSmithDocs'

In [15]:
tools = [wiki, arvix, retriever_tool]

In [16]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\matcg\\OneDrive\\Documents\\GitHub\\LangChain Projects\\4-Search Engine\\venv_se\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='LangSmithDocs', description='useful for when you need to answer questions about LangSmithDocs', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000024BF552A1F0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000024BF552A

### Run all this tools with Agents and LLM Models

#### Tools, LLM --> AgentExecutor

In [48]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
import openai
groq_api_key = os.getenv("GROQ_API_KEY")
openai.api_key = os.getenv("OPENAI_API_KEY")

# llm=ChatGroq(groq_api_key=groq_api_key, model="llama-3.3-70b-versatile")
llm = ChatOpenAI(model="gpt-5.2")

In [49]:
## Prompt Template (Another way to create a prompt template)
from langchain_classic import hub
prompt=hub.pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [50]:
## Agents
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(
    llm,
    tools,
    prompt
)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat

In [51]:
## Agent Executor
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='

In [52]:
agent_executor.invoke({"input": "Tell me about LangSmith"})



> Entering new AgentExecutor chain...

Invoking: `LangSmithDocs` with `{'query': 'LangSmith overview what is LangSmith'}`


LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before explori

{'input': 'Tell me about LangSmith',
 'output': 'LangSmith is a framework-agnostic platform (by LangChain) for building, debugging, evaluating, and deploying LLM apps and AI agents. It’s designed to sit alongside whatever stack you’re using (LangChain, custom code, different model providers) and give you a single place to understand what your system is doing and improve it over time.\n\nKey parts:\n\n- **Observability / Tracing**\n  - Records end-to-end traces of requests through your app/agent (calls to models, tools, retrievers, prompts, intermediate steps).\n  - Helps you debug failures, latency, unexpected tool calls, prompt issues, and flaky behavior.\n\n- **Evaluation**\n  - Run evaluations on outputs to measure quality and track it over time.\n  - Common uses: regression testing prompt/model changes, scoring responses (LLM-as-judge or custom metrics), comparing versions/experiments.\n\n- **Prompt engineering workflow**\n  - Tools for iterating on prompts with versioning and coll

Usou o Tool do LangSmith do WebLoader!!!!

In [53]:
agent_executor.invoke({"input": "What is Machine Learning?"})



> Entering new AgentExecutor chain...
Machine Learning (ML) is a branch of artificial intelligence where computers learn patterns from data to make predictions or decisions, instead of being explicitly programmed with fixed rules.

### Core idea
- You provide **examples** (data), often with **answers** (labels).
- The algorithm finds a **model** (a mathematical function) that maps inputs → outputs.
- The model can then **generalize** to new, unseen data.

### Common types of machine learning
1. **Supervised learning**: Learn from labeled data (e.g., email → spam/not spam, house features → price).
2. **Unsupervised learning**: Find structure in unlabeled data (e.g., clustering customers into segments).
3. **Reinforcement learning**: Learn by trial and error with rewards (e.g., game-playing agents, robotics control).

### Typical applications
- Recommendation systems (movies, products)
- Image and speech recognition
- Fraud detection
- Medical risk prediction
- Language models and chat

{'input': 'What is Machine Learning?',
 'output': 'Machine Learning (ML) is a branch of artificial intelligence where computers learn patterns from data to make predictions or decisions, instead of being explicitly programmed with fixed rules.\n\n### Core idea\n- You provide **examples** (data), often with **answers** (labels).\n- The algorithm finds a **model** (a mathematical function) that maps inputs → outputs.\n- The model can then **generalize** to new, unseen data.\n\n### Common types of machine learning\n1. **Supervised learning**: Learn from labeled data (e.g., email → spam/not spam, house features → price).\n2. **Unsupervised learning**: Find structure in unlabeled data (e.g., clustering customers into segments).\n3. **Reinforcement learning**: Learn by trial and error with rewards (e.g., game-playing agents, robotics control).\n\n### Typical applications\n- Recommendation systems (movies, products)\n- Image and speech recognition\n- Fraud detection\n- Medical risk prediction

In [54]:
agent_executor.invoke({"input": "What is the paper 	arXiv:1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': 'arXiv:1706.03762'}`


Published: 2020-02-12
Title: GLU Variants Improve Transformer
Authors: Noam Shazeer
Summary: Gated Linear Units (arXiv:1612.08083) consist of the component-wise product of two linear projections, one of which is first passed through a sigmoid functioarXiv:1706.03762 is **“Attention Is All You Need”** (Vaswani et al., 2017), the paper that introduced the **Transformer** architecture.

- **Core idea:** Replace recurrence (RNNs) and convolution with **self-attention** to model relationships between tokens in a sequence.
- **Architecture:** An **encoder–decoder** model built from repeated blocks of:
  - **Multi-head self-attention** (lets the model attend to different relationship types in parallel)
  - **Position-wise feed-forward networks**
  - **Residual connections + layer normalization**
  - **Positional encodings** to inject word order information (since there’s no recurrence).
- **Why i

{'input': 'What is the paper \tarXiv:1706.03762 about?',
 'output': 'arXiv:1706.03762 is **“Attention Is All You Need”** (Vaswani et al., 2017), the paper that introduced the **Transformer** architecture.\n\n- **Core idea:** Replace recurrence (RNNs) and convolution with **self-attention** to model relationships between tokens in a sequence.\n- **Architecture:** An **encoder–decoder** model built from repeated blocks of:\n  - **Multi-head self-attention** (lets the model attend to different relationship types in parallel)\n  - **Position-wise feed-forward networks**\n  - **Residual connections + layer normalization**\n  - **Positional encodings** to inject word order information (since there’s no recurrence).\n- **Why it mattered:** Self-attention enables much more **parallel computation** than RNNs and captures **long-range dependencies** effectively.\n- **Results:** Achieved state-of-the-art performance on machine translation benchmarks (notably WMT’14 English–German and English–Fren

Usou o Tool do Arxiv!!!!